# Week 11 Lab: Decision Trees, Random Forests, and Feature Importance

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-11-trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this lab, you'll build decision trees and random forests, compare how well they generalize, and then explain what your best model learned using permutation importance and partial dependence plots.

## 🎯 Learning Objectives
By the end of this lab, you will be able to:
- Build decision tree and random forest models with scikit-learn
- Use the gap between training and test performance to spot overfitting
- Compare random forest settings and select a model based on the metric that matters most
- Explain a model with permutation importance and partial dependence plots

## 📚 This Lab Reinforces
- **Chapter 25: Decision Trees**
- **Chapter 26: Random Forests**
- **Chapter 27: Understanding Feature Importance**
- **Week 11 Tuesday Slides: Tree-Based Models**

## 🕐 Estimated Time & Structure
**Total Time:** 75 minutes
**Mode:** Work together in groups (2–4 students), submit individually

- **[0–25 min]** Guided review, experiments, and practice with the Boston housing data
- **[25–30 min]** Class discussion and Q&A
- **[30–70 min]** Independent challenges with the Breast Cancer Wisconsin dataset
- **[70–75 min]** Lab wrap-up and homework quiz preparation

You are encouraged to work through the lab with **2–4 classmates**: talk through the problems and help each other get past coding hurdles. However, **you must write your own code, and you will submit your own notebook and homework quiz.**

## 💡 Why This Matters
Random forests are one of the most widely used models in business because they capture non-linear patterns and interactions without much tuning. But a model nobody can explain rarely gets used. This lab pairs the two skills: building an accurate forest, then explaining what drives its predictions.

## Setup

We'll work with two datasets: the **Boston** housing data from ISLP for guided practice (the same data from Tuesday's lecture), and the **Breast Cancer Wisconsin** dataset for the independent challenges.

The `ISLP` package isn't installed in Colab by default, so run this cell first.

> ⏳ The install takes about a minute. If Colab asks you to **restart the session**, click **Restart**, then continue with the next cell (you don't need to rerun this one).

In [ ]:
%pip install -q ISLP

In [ ]:
# Required imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    root_mean_squared_error, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# Use this random state everywhere so your results match the homework quiz
RANDOM_STATE = 42

# Load the datasets
Boston = load_data('Boston')
cancer = pd.read_csv("https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/breast_cancer.csv")

print("Boston dataset shape:", Boston.shape)
print("Cancer dataset shape:", cancer.shape)

## Part 1 — Guided Review: Boston Housing Prices (25 minutes)

Let's work through the tree-based workflow from Tuesday's lecture. The target is `medv`, the median home value in each neighborhood (in $1,000s).

**📋 Workflow:**
1. Split the data
2. Fit a single decision tree, then experiment with `max_depth`
3. Fit a random forest, then experiment with `n_estimators` and `max_depth`
4. Fit a final forest and find which features matter most
5. Use a partial dependence plot to see *how* the top feature moves predictions

Run each cell, and when you see 🔧 **Try it**, change the values and re-run the cell.

In [ ]:
# Step 1: Separate features and target, then split 70/30
X_boston = Boston.drop(columns='medv')
y_boston = Boston['medv']

X_train, X_test, y_train, y_test = train_test_split(
    X_boston, y_boston, test_size=0.3, random_state=RANDOM_STATE
)

print(f"Training set: {len(X_train)} neighborhoods")
print(f"Test set:     {len(X_test)} neighborhoods")

### Step 2: A Single Decision Tree

`max_depth` limits how many questions deep the tree can go. We'll start with a shallow tree and evaluate it on both the training and test data.

In [ ]:
# Step 2: Fit a single decision tree and evaluate it
tree_depth = 3   # 🔧 change this value and re-run the cell

dt_model = DecisionTreeRegressor(max_depth=tree_depth, random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

print(f"max_depth = {tree_depth}")
print(f"Train R²:  {dt_model.score(X_train, y_train):.3f}")
print(f"Test R²:   {dt_model.score(X_test, y_test):.3f}")
print(f"Test RMSE: ${root_mean_squared_error(y_test, dt_model.predict(X_test)):.2f}K")

🔧 **Try it (2–3 minutes):** Re-run the cell above with `tree_depth` set to **1, 2, 5, 8, and `None`** (no limit). Note the train and test R² for each.

- What happens to **training** R² as the tree gets deeper?
- Does **test** R² follow the same pattern?

*Your notes:*

### Step 3: A Random Forest

You probably noticed that training R² climbs with every extra level of depth (all the way to 1.000 with no limit), but test R² **bounces around** instead of improving steadily. A single tree is unstable: one extra split can change its predictions a lot. A random forest averages many trees to smooth that out.

`n_estimators` sets how many trees are in the forest, and `max_depth` limits how deep each tree can grow.

In [ ]:
# Step 3: Fit a random forest and evaluate it
n_trees = 100      # 🔧 try 10, 50, 500
forest_depth = 3   # 🔧 try 5, 10, None

rf_model = RandomForestRegressor(
    n_estimators=n_trees, max_depth=forest_depth, random_state=RANDOM_STATE
)
rf_model.fit(X_train, y_train)

print(f"n_estimators = {n_trees}, max_depth = {forest_depth}")
print(f"Train R²:  {rf_model.score(X_train, y_train):.3f}")
print(f"Test R²:   {rf_model.score(X_test, y_test):.3f}")
print(f"Test RMSE: ${root_mean_squared_error(y_test, rf_model.predict(X_test)):.2f}K")

🔧 **Try it (3–4 minutes):** Change **one setting at a time** and re-run the cell above.

- Keep `forest_depth = 3` and try `n_trees` of **10, 50, and 500**. How much does the number of trees change test R²?
- Keep `n_trees = 100` and try `forest_depth` of **5, 10, and `None`**. Which setting matters more for this data?
- Did any forest beat your best single tree on the test data?

*Your notes:*

### Step 4: A Final Forest — Which Features Matter?

Two patterns usually show up in the experiments above: **depth matters a lot** (deeper forests reach a test R² of about 0.87, compared to about 0.81 at depth 3), and past about 50 trees, **more trees barely change the results**; they just take longer to train. So our final forest uses **500 trees with no depth limit**.

In [ ]:
# Step 4a: Fit the final random forest
final_rf = RandomForestRegressor(n_estimators=500, max_depth=None, random_state=RANDOM_STATE)
final_rf.fit(X_train, y_train)

print(f"Train R²:  {final_rf.score(X_train, y_train):.3f}")
print(f"Test R²:   {final_rf.score(X_test, y_test):.3f}")
print(f"Test RMSE: ${root_mean_squared_error(y_test, final_rf.predict(X_test)):.2f}K")

In [ ]:
# Step 4b: Gini importance (built in) vs. permutation importance (on the test set)
perm_result = permutation_importance(
    final_rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='r2'
)

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'gini_importance': final_rf.feature_importances_,
    'permutation_importance': perm_result.importances_mean
}).sort_values('permutation_importance', ascending=False)

importance_df.head()

**Reading the importances:**

- **Gini importance** credits each feature with the impurity drop from every split that uses it, on the **training** data.
- **Permutation importance** shuffles one column of the **test** data at a time and measures how much test R² drops. Here, shuffling `lstat` drops R² by about 0.52.
- Both put `lstat` (% lower-status residents) and `rm` (average rooms) far ahead of everything else. Gini nearly ties them; permutation importance puts `lstat` clearly first. When the two disagree, trust permutation importance for "what matters on new data".

### Step 5: How Does the Top Feature Move Predictions?

A partial dependence plot sets every neighborhood to the same value of one feature, keeps everything else as is, and averages the forest's predictions.

In [ ]:
# Step 5: Partial dependence plot for the most important feature
top_feature = importance_df.iloc[0]['feature']

fig, ax = plt.subplots(figsize=(8, 4))
pdp = PartialDependenceDisplay.from_estimator(
    final_rf, X_train, features=[top_feature], kind='both',
    subsample=50, random_state=RANDOM_STATE, ax=ax
)
pdp.axes_[0, 0].set_ylabel('Predicted value ($1,000s)')
plt.show()

**Reading the PDP:** each thin line is one neighborhood's "what-if" curve as `lstat` changes, and the dashed line is their **average**. Predicted home values fall from about $32K to about $15K as `lstat` rises, then level off once `lstat` passes roughly 20%. That leveling off is a pattern a single linear coefficient would miss.

### 🧪 Practice Exercise — Shuffle a Column by Hand

**Business Scenario:** A real estate analyst doesn't trust the permutation importance numbers. Show them what's happening under the hood by shuffling `lstat` yourself.

**Your Task:** Fill in the `___` blanks.

1. Make a copy of the test data and shuffle its `lstat` column
2. Score the final forest on the original test data and on the shuffled copy
3. How much did test R² drop? Is it close to the permutation importance for `lstat` in Step 4? Why might it not match exactly?

In [ ]:
# Step 1: Copy the test data and shuffle one column
X_test_shuffled = X_test.copy()
rng = np.random.default_rng(RANDOM_STATE)
X_test_shuffled['lstat'] = rng.permutation(X_test_shuffled['___'].to_numpy())

# Step 2: Score the final forest on the original and the shuffled test data
original_r2 = final_rf.score(___, ___)
shuffled_r2 = final_rf.score(___, ___)

print(f"Original test R²: {original_r2:.3f}")
print(f"Shuffled test R²: {shuffled_r2:.3f}")
print(f"Drop:             {original_r2 - shuffled_r2:.3f}")

*Step 3 — Your comparison:*

## Class Discussion/Q&A (5 minutes)

**Discussion prompts:**
- Your single trees' test R² jumped around as you changed `max_depth`, but the forests' didn't. In your own words, why does averaging many trees help?
- Gini importance nearly tied `lstat` and `rm`, while permutation importance put `lstat` clearly first. Which would you report to a manager, and why?
- How would you explain the `lstat` PDP to someone who has never heard of a random forest?

**Common blockers and clarifications:**
- `.score()` returns R² for regression models and **accuracy** for classification models
- Fit on the **training** data; compute permutation importance on the **test** data
- For a classification model, a PDP's y-axis is the **predicted probability** of the positive class

## Part 2 — Independent Challenges: Breast Cancer Diagnosis (40 minutes)

For the next three challenges, you'll return to the breast cancer data from Week 10, this time with trees and forests. The specific numerical results you get will be used to answer questions on this week's **Canvas homework quiz**.

* **The support decreases with each challenge.** Challenge 1 gives you starter code with blanks, Challenge 2 gives you an outline only, and Challenge 3 is on your own.
* **Work together, submit individually.** Talk through the challenges with your group and help each other get unstuck, but **write your own code**. You'll each submit your own notebook and quiz.
* **DO NOT USE AI** to generate code for you.
* Feel free to ask questions or seek help from the instructor. Periodically, we'll stop and walk through parts of a challenge together.
* **Important:** use the exact parameters given (`test_size=0.3`, `random_state=RANDOM_STATE`, `stratify=y_cancer`, and the model settings in each challenge). These produce the exact numbers needed for the quiz. Steps marked **📝** feed directly into quiz questions.

### 🔬 About the Dataset

The **Breast Cancer Wisconsin (Diagnostic)** dataset describes 569 breast tissue samples. Each of 10 cell nucleus measurements (such as `radius`, `texture`, `area`, and `concavity`) is summarized three ways: **`_mean`** (the average), **`_se`** (the standard error), and **`_worst`** (the average of the three largest values). The target, `diagnosis`, is **`M`** (malignant) or **`B`** (benign).

**Business Context:** Missing a malignant tumor (a **false negative**) delays treatment and can be life-threatening. A false alarm leads to an unnecessary follow-up biopsy. So for this hospital, **recall** (catching malignant cases) matters most.

> ⚠️ **Watch for:** some column names contain a space, such as `concave points_worst`. Copy column names exactly.

In [ ]:
# Prepare the target and features (PROVIDED)
y_cancer = (cancer['diagnosis'] == 'M').astype(int)   # 1 = malignant, 0 = benign
X_cancer = cancer.drop(columns='diagnosis')            # all 30 numeric measurements

print(f"Features: {X_cancer.shape[1]}")
print(f"Malignant rate: {y_cancer.mean():.1%}")

### Challenge 1 — Single Decision Trees

🧩 **Starter code provided:** fill in the `___` blanks.

**Business Question:** How deep should a single decision tree be to classify these tumors well?

**Your Task:**

1. **Split the data**: 70/30 train/test split with `random_state=RANDOM_STATE` and `stratify=y_cancer`. 📝 Which `train_test_split()` parameter keeps the malignant rate the same in both sets?
2. **Fit trees at three depths**: For `max_depth` of **3, 5, and 10**, fit a `DecisionTreeClassifier` (with `random_state=RANDOM_STATE`) and record its **training** and **test** accuracy.
3. **Plot the results**: training and test accuracy against `max_depth`.
4. **Interpret**: 📝 Which depth has the best **test** accuracy? 📝 What does the depth-10 tree's train/test gap tell you?

In [ ]:
# Step 1: Split the data (70/30, random_state=RANDOM_STATE, stratify=y_cancer)
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
)

print(f"Training set: {len(X_train_cancer)} samples ({y_train_cancer.mean():.1%} malignant)")
print(f"Test set:     {len(X_test_cancer)} samples ({y_test_cancer.mean():.1%} malignant)")

In [ ]:
# Step 2: Fit a tree at each depth and record training and test accuracy
tree_results = []
for depth in [3, 5, 10]:
    tree = DecisionTreeClassifier(max_depth=___, random_state=RANDOM_STATE)
    tree.fit(___, ___)
    tree_results.append({
        'max_depth': depth,
        'train_accuracy': tree.score(___, ___),
        'test_accuracy': tree.score(___, ___)
    })

tree_results = pd.DataFrame(tree_results)
tree_results

In [ ]:
# Step 3: Plot training and test accuracy against max_depth
plt.plot(tree_results['max_depth'], tree_results['___'], marker='o', label='Training accuracy')
plt.plot(tree_results['max_depth'], tree_results['___'], marker='o', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

*Step 4 — Your interpretation:*

### Challenge 2 — Random Forests

📝 **Outline only:** the code cell lists each step as a comment. You write all the code.

**Business Question:** Can a random forest beat a single tree, and which forest settings should the hospital use?

**Your Task:**

1. **Fit four forests** with the settings below (all with `random_state=RANDOM_STATE`):

    | Model | n_estimators | max_depth | max_features |
    |-------|--------------|-----------|--------------|
    | RF1   | 100          | 10        | `'sqrt'`     |
    | RF2   | 200          | 15        | `'sqrt'`     |
    | RF3   | 300          | 20        | 10           |
    | RF4   | 500          | None      | `'sqrt'`     |

2. **Evaluate each forest**: training accuracy, plus test accuracy, precision, recall, and F1-score. 📝 Which two forests have the highest **recall**?
3. **Compare with the trees**: How do the forests' test accuracies compare with your best tree from Challenge 1? 📝 Why do random forests tend to outperform a single tree?
4. **Select the best model**: Pick the forest with the **highest recall**. If two are tied, choose the one with **fewer trees** (it's faster and simpler, with the same performance). Save it as **`best_rf`**; you'll use it in Challenge 3. 📝 Why is recall the right metric for this choice?
5. **Explain a setting**: 📝 In your own words, what does `max_features='sqrt'` control?

In [ ]:
# Step 1: The four forest configurations (PROVIDED)
rf_configs = {
    'RF1': {'n_estimators': 100, 'max_depth': 10, 'max_features': 'sqrt'},
    'RF2': {'n_estimators': 200, 'max_depth': 15, 'max_features': 'sqrt'},
    'RF3': {'n_estimators': 300, 'max_depth': 20, 'max_features': 10},
    'RF4': {'n_estimators': 500, 'max_depth': None, 'max_features': 'sqrt'},
}

# Step 2: Loop over rf_configs. For each one, fit
#         RandomForestClassifier(**params, random_state=RANDOM_STATE) on the training data
#         (**params unpacks the dictionary into keyword arguments), then record training
#         accuracy, test accuracy, precision, recall, and F1-score

# Step 3: Put the results in a DataFrame and compare test accuracy with your best tree

# Step 4: Save the selected forest as best_rf (highest recall; if tied, fewer trees)


*Steps 3–5 — Your answers:*

### Challenge 3 — Explain the Model

🚀 **On your own:** no starter code. Use `best_rf` from Challenge 2.

**Business Question:** Before the hospital adopts your model, the physicians want to know **which** measurements drive its predictions and **how** the most important one changes the predicted risk.

**Your Task:**

1. **Permutation importance**: Calculate permutation importance for `best_rf` on the **test** set with `n_repeats=10` and `random_state=RANDOM_STATE`. Leave `scoring` at its default (accuracy for a classifier). 📝 Which feature is the most important?
2. **Top 5 features**: Show the top 5 features in a horizontal bar chart. 📝 What do all five have in common?
3. **Partial dependence plot**: Create a PDP for the #1 feature. 📝 What does the PDP's y-axis represent?
4. **Explain it to the physicians**: In 2–3 sentences, describe which measurements matter most and how the #1 feature changes the predicted probability of malignancy (is there a point where it rises sharply?). Add one caution about what feature importance does *not* tell them.

💡 **Hint:** Revisit Part 1 (Steps 4–5) and the "Which Features Matter, and How?" section of Tuesday's lecture notebook.

In [ ]:
# Your code here

*Step 4 — Your explanation for the physicians:*

## 🎓 Lab Wrap-Up & Reflection

### ✅ What You Accomplished
In this lab, you practiced:
- Building decision trees and spotting overfitting from the train/test gap
- Comparing random forest settings and selecting a model by the metric that matters most
- Measuring feature importance by shuffling columns and reading partial dependence plots
- Translating a model's behavior into an explanation a non-technical audience can use

### 🤔 Reflection Questions
Take 2-3 minutes to consider:
- The deepest tree had perfect training accuracy but the worst test accuracy. What would you tell a teammate who only reported training accuracy?
- Several forests had identical recall. Besides speed, what else might you consider when choosing between them?
- Your top features were all measurements of the most abnormal cells. Why might that make sense to a pathologist?

### 🔗 Connection to Course Goals
A model only creates value if people trust it enough to act on it. This lab pairs building an accurate model with explaining it, which is what turns a prediction into a decision someone is willing to make.

### 📋 Next Steps
- **Homework:** Complete the Canvas quiz using your numerical results from today's challenges, and upload your completed notebook (due Sunday)
- **Next Tuesday:** Cross-validation and hyperparameter tuning: choosing settings like `max_depth` systematically instead of by hand
- **Additional Practice:** Try the end-of-chapter exercises in Chapters 25–27

---
**💾 Save your work.** You'll upload this notebook with your quiz, so make sure every cell has been run and shows its output.

## 🚨 Troubleshooting & Common Issues

**Issue 1:** `KeyError: 'concave_points_worst'`
- **Solution:** The column name has a space, not an underscore: `'concave points_worst'`. Use `X_cancer.columns` to see the exact names.

**Issue 2:** My numbers don't match my groupmates'
- **Solution:** Check that you used `test_size=0.3`, `random_state=RANDOM_STATE`, and `stratify=y_cancer` in the split, and `random_state=RANDOM_STATE` in every model.

**Issue 3:** My most important feature is different from my groupmates'
- **Solution:** Make sure you computed permutation importance for `best_rf` (selected with the tie-breaking rule), on the **test** set, with `n_repeats=10`, `random_state=RANDOM_STATE`, and no `scoring` argument.

**Issue 4:** Some permutation importances are slightly negative
- **Solution:** That's normal. Shuffling an unimportant feature can improve performance slightly by chance. Treat those features as unimportant.

**Issue 5:** `TypeError` when passing `**params`
- **Solution:** `**params` must come from one of the dictionaries in `rf_configs`, for example `for name, params in rf_configs.items():`.

**General Debugging Tips:**
- Fit on the **training** data; evaluate on the **test** data
- If a variable seems wrong, re-run the notebook from the top (Runtime → Run all) to clear out stale values